# Project 6: Test-Driven Code-Repair Agent

Compare a one-shot patch with a three-attempt repair loop using AST mapping,
failure localization, constrained unified diffs, public and hidden tests, and
exact rollback. Gold corrected programs never enter planner context.

In [1]:
from pathlib import Path
import os,subprocess,sys
candidates=[Path.cwd(),Path.cwd()/"project6",Path("/content/ai_agentic_attemptings/project6")]
PROJECT_ROOT=next((p.resolve() for p in candidates if (p/"config/default.json").exists()),None)
if PROJECT_ROOT is None:
    repo=Path("/content/ai_agentic_attemptings")
    if not repo.exists(): subprocess.run(["git","clone","https://github.com/soraber/ai_agentic_attemptings.git",str(repo)],check=True)
    else: subprocess.run(["git","-C",str(repo),"pull","--ff-only"],check=True)
    PROJECT_ROOT=repo/"project6"
os.chdir(PROJECT_ROOT)
if not os.getenv("AI_PROJECT_SKIP_INSTALL"):
    subprocess.run([sys.executable,"-m","pip","install","--upgrade-strategy","only-if-needed","-r","requirements-colab.txt"],check=True)
    subprocess.run([sys.executable,"-m","pip","install","-e",".","--no-deps"],check=True)
source_root=PROJECT_ROOT/"src"
if str(source_root) not in sys.path: sys.path.insert(0,str(source_root))
if not os.getenv("AI_PROJECT_SKIP_INSTALL"):
    check=subprocess.run([sys.executable,"-m","pip","check"],text=True,capture_output=True)
    if check.returncode: print(check.stdout or check.stderr)
from project6_agent.agent import RepairAgent
try: import torch
except ImportError: torch=None
print("Project 6 imports passed")
print({"cuda":bool(torch and torch.cuda.is_available()),"gpu":torch.cuda.get_device_name(0) if torch and torch.cuda.is_available() else None})

ipython 7.34.0 requires jedi, which is not installed.
ibis-framework 9.5.0 has requirement sqlglot<25.21,>=23.4, but you have sqlglot 29.0.1.

Project 6 imports passed
{'cuda': True, 'gpu': 'NVIDIA A100-SXM4-40GB'}


In [2]:
import getpass,os,sys
from project6_agent.config import load_config
EVAL_BACKEND="local_gpu"  # openai | local_gpu
RUN_FULL_EVAL=True; RUN_API_EVAL=EVAL_BACKEND=="openai"; RUN_LOCAL_GPU_EVAL=EVAL_BACKEND=="local_gpu"
config=load_config(PROJECT_ROOT/"config/default.json")
if RUN_API_EVAL and not os.getenv("OPENAI_API_KEY"):
    if "google.colab" in sys.modules:
        from google.colab import userdata
        key=userdata.get("OPENAI_API_KEY")
    else: key=getpass.getpass("OPENAI_API_KEY (hidden): ")
    if not key: raise RuntimeError("OPENAI_API_KEY required for API mode")
    os.environ["OPENAI_API_KEY"]=key
if RUN_LOCAL_GPU_EVAL:
    import gc,torch
    if not torch.cuda.is_available(): raise RuntimeError("Select a Colab GPU runtime for local_gpu mode")
    stale_names=("source_planner","schema_retriever","local_backend","planners","planner","embedding_retriever","local_answerer")
    released=[name for name in stale_names if globals().pop(name,None) is not None]
    gc.collect(); torch.cuda.empty_cache(); free_bytes,total_bytes=torch.cuda.mem_get_info()
    print({"released_gpu_objects":released,"free_gpu_gib":round(free_bytes/2**30,2),"total_gpu_gib":round(total_bytes/2**30,2)})
print(config.model_dump())

{'released_gpu_objects': [], 'free_gpu_gib': 39.08, 'total_gpu_gib': 39.49}
{'project_id': 'project6', 'seed': 20260802, 'model': 'gpt-5.6-sol', 'reasoning_effort': 'medium', 'max_patch_attempts': 3, 'max_changed_lines': 30, 'test_timeout_seconds': 20.0, 'max_test_output_chars': 12000, 'max_model_calls': 220, 'max_output_tokens': 900, 'max_retries': 2, 'max_estimated_cost_usd': 8.0, 'input_price_per_million_usd': 5.0, 'output_price_per_million_usd': 30.0, 'local_model': 'Qwen/Qwen2.5-Coder-7B-Instruct', 'local_device': 'cuda', 'local_max_new_tokens': 900, 'development_case_count': 4, 'test_case_count': 8}


In [3]:
import subprocess,sys
subprocess.run([sys.executable,"tools/fetch_quixbugs.py"],check=True)
QUIXBUGS_ROOT=PROJECT_ROOT/"data/cache/QuixBugs"

In [4]:
import json
manifest=json.loads((PROJECT_ROOT/"data/quixbugs_manifest.json").read_text())
development_cases=[c for c in manifest["cases"] if c["split"]=="development"]
test_cases=[c for c in manifest["cases"] if c["split"]=="test"]
assert (len(development_cases),len(test_cases))==(4,8)
print({"commit":manifest["commit"],"development":4,"test":8})

{'commit': '4257f44b0ff1181dedaedee6a447e133219fcebf', 'development': 4, 'test': 8}


In [5]:
print("One-shot fixture behavior is covered by tests/test_agent.py; run P06-C07 before API patches.")

One-shot fixture behavior is covered by tests/test_agent.py; run P06-C07 before API patches.


In [6]:
from project6_agent.repository import build_repository_map
symbols=build_repository_map(QUIXBUGS_ROOT/"python_programs")
print({"mapped_symbols":len(symbols),"example":symbols[0].model_dump()})

{'mapped_symbols': 52, 'example': {'path': 'bitcount.py', 'name': 'bitcount', 'kind': 'function', 'line': 2, 'imports': [], 'docstring': None}}


In [7]:
import os,subprocess,sys
test_env=os.environ.copy(); test_env["PYTEST_DISABLE_PLUGIN_AUTOLOAD"]="1"
result=subprocess.run([sys.executable,"-m","pytest","-q","tests"],text=True,capture_output=True,env=test_env,timeout=120); print(result.stdout)
if result.returncode: print(result.stderr); raise RuntimeError("Project 6 tests failed")

.........                                                                [100%]
9 passed in 2.69s



In [8]:
from project6_agent.agent import RepairAgent
from project6_agent.evaluation import summarize_results
from project6_agent.planners import LocalPatchPlanner, OpenAIPatchPlanner, TransformersPatchBackend
from project6_agent.quixbugs import prepare_case
if not RUN_FULL_EVAL:
    print("Set RUN_FULL_EVAL=True and RUN_API_EVAL=True after P06-C07 passes.")
else:
    result_dir=PROJECT_ROOT/("output/gpu" if RUN_LOCAL_GPU_EVAL else "output")
    results=[]; planners=[]; agent=RepairAgent(config); runtime=result_dir/"runtime/quixbugs"; runtime.mkdir(parents=True,exist_ok=True)
    local_backend=TransformersPatchBackend(config) if RUN_LOCAL_GPU_EVAL else None
    for case in test_cases:
        prepared=prepare_case(QUIXBUGS_ROOT,case,runtime/f"source-{case['bug_id']}")
        for system in ["one_shot","repair_loop"]:
            if RUN_LOCAL_GPU_EVAL:
                planner=LocalPatchPlanner(config,prepared["root"]/prepared["source"],case["bug_id"],prepared["source"],local_backend)
            else:
                planner=OpenAIPatchPlanner(config,prepared["root"]/prepared["source"],case["bug_id"],allowed_path=prepared["source"])
            planners.append(planner)
            results.append(agent.run(case["bug_id"],system,prepared["root"],runtime/f"work-{case['bug_id']}-{system}",[prepared["source"]],prepared["public_tests"],prepared["hidden_tests"],planner))
    usage=local_backend.usage_summary() if RUN_LOCAL_GPU_EVAL else {"model_calls":sum(p.calls for p in planners),"input_tokens":sum(p.input_tokens for p in planners),"output_tokens":sum(p.output_tokens for p in planners),"estimated_cost_usd":round(sum(p.estimated_cost_usd for p in planners),6)}
    selected_model=config.local_model if RUN_LOCAL_GPU_EVAL else config.model
    summary=summarize_results(results,result_dir,planner_usage=usage,model=selected_model,evaluation_backend=EVAL_BACKEND); print(summary)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

{'project': 'Test-Driven Code-Repair Agent', 'result_status': 'measured', 'model': 'Qwen/Qwen2.5-Coder-7B-Instruct', 'evaluation_backend': 'local_gpu', 'planner_usage': {'model_calls': 32, 'input_tokens': 27588, 'output_tokens': 8326, 'estimated_cost_usd': 0.0, 'local_model': 'Qwen/Qwen2.5-Coder-7B-Instruct', 'device': 'cuda'}, 'one_shot': {'cases': 8, 'verified_repair_rate_pct': 0.0, 'hidden_pass_rate_pct': 0.0, 'overfit_rate_pct': 0.0, 'rollback_success_pct': 100.0, 'mean_changed_lines': 6.625, 'median_latency_seconds': 10.49912349899978}, 'repair_loop': {'cases': 8, 'verified_repair_rate_pct': 0.0, 'hidden_pass_rate_pct': 0.0, 'overfit_rate_pct': 0.0, 'rollback_success_pct': 100.0, 'mean_changed_lines': 4.875, 'median_latency_seconds': 23.32945965350018}}


In [9]:
import json
result_dir=PROJECT_ROOT/("output/gpu" if RUN_LOCAL_GPU_EVAL else "output")
path=result_dir/"project6_representative_samples.json"
print(json.loads(path.read_text()) if path.exists() else "Run the paired evaluator first.")

{'failed': [{'attempts': 1, 'bug_id': 'gcd', 'changed_lines': 6, 'error': 'PermissionError: patch must be a complete unified diff', 'hidden_passed': False, 'latency_seconds': 8.06591438400028, 'overfit_detected': False, 'public_passed': False, 'rollback_verified': True, 'system': 'one_shot', 'trajectory': ['map_repository', 'run_public_tests', 'localize_failure', 'plan_patch', 'reject_patch', 'rollback'], 'verified': False}, {'attempts': 3, 'bug_id': 'gcd', 'changed_lines': 2, 'error': 'PermissionError: patch must be a complete unified diff', 'hidden_passed': False, 'latency_seconds': 15.32757197400042, 'overfit_detected': False, 'public_passed': False, 'rollback_verified': True, 'system': 'repair_loop', 'trajectory': ['map_repository', 'run_public_tests', 'localize_failure', 'plan_patch', 'reject_patch', 'rollback', 'plan_patch', 'reject_patch', 'rollback', 'plan_patch', 'reject_patch', 'rollback'], 'verified': False}, {'attempts': 1, 'bug_id': 'get_factors', 'changed_lines': 3, 'erro

In [10]:
import subprocess,sys
result_dir=PROJECT_ROOT/("output/gpu" if RUN_LOCAL_GPU_EVAL else "output"); summary_path=result_dir/"project6_final_summary.json"
if summary_path.exists():
    subprocess.run([sys.executable,"tools/generate_report.py","--summary",str(summary_path),"--output",str(result_dir/"project6_report.pdf")],check=True)
    subprocess.run([sys.executable,"tools/validate_project.py"]+([] if RUN_LOCAL_GPU_EVAL else ["--require-results"]),check=True)
else: print("Measured summary absent; report generation skipped.")